# VectorBT Pro and OSS on Current Case-Study Strategies

This notebook reports the VectorBT Pro and VectorBT OSS rows from the current real-strategy audit.
Required comparisons use ETF, CME futures, USD-quoted foreign-exchange, and US equity-panel target
streams where the pinned VectorBT edition supports the asset and accounting contract.

**Learning objectives**

- Compare VectorBT Pro and OSS with ML4T on supported real-data workloads
- Distinguish fill precision from the monetary unit used for account values
- Understand why VectorBT OSS is not used for the CME futures contract
- Read engine-only runtime evidence without treating it as a universal ranking

**Book reference**: Chapter 16, Section 16.3

<!-- ML4T_ZH_BEGIN c0000-p000 -->
# 目前案例研究策略的 VectorBT Pro 與開源版比較

本 Notebook 報告目前真實策略稽核中的 VectorBT Pro 與 VectorBT 開源版（OSS）資料列。必要比較使用 ETF、CME 期貨、以美元報價的外匯，以及美國股票資料面板的目標序列；前提是固定版本的 VectorBT 能支援該資產與會計規格。

**學習目標**

- 在受支援的真實資料工作負載上，比較 VectorBT Pro、開源版與 ML4T
- 區分成交精度與帳戶價值所使用的貨幣單位
- 了解為何 CME 期貨契約不使用 VectorBT 開源版
- 判讀僅計引擎的執行時間證據，而不視為通用速度排名

**書籍參考**：第 16 章第 16.3 節
<!-- ML4T_ZH_END c0000-p000 -->


## Setup

<!-- ML4T_ZH_BEGIN c0001-p000 -->
## 設定
<!-- ML4T_ZH_END c0001-p000 -->


In [1]:
"""Current VectorBT parity evidence."""

import json

import polars as pl
from IPython.display import display

from utils.paths import get_chapter_dir

In [2]:
# Production defaults - Papermill injects overrides after this cell
ROUND_SECONDS = 3

In [3]:
AUDIT_PATH = get_chapter_dir(16) / "resources" / "framework_parity_audit.json"
audit = json.loads(AUDIT_PATH.read_text(encoding="utf-8"))
FRAMEWORKS = ["vectorbt_pro", "vectorbt_oss"]
FRAMEWORK_NAMES = {
    key: f"{audit['frameworks'][key]['display_name']} {audit['frameworks'][key]['version']}"
    for key in FRAMEWORKS
}
CASE_NAMES = {
    "etfs": "ETF allocation",
    "cme_futures": "CME futures",
    "crypto_perps_funding": "Crypto perpetual funding",
    "fx_pairs": "FX allocation (USD-quoted pairs)",
    "us_equities_panel": "US equity panel",
}

## 1. Required comparisons

<!-- ML4T_ZH_BEGIN c0005-p000 -->
## 1. 必要比較
<!-- ML4T_ZH_END c0005-p000 -->


In [4]:
results = (
    pl.DataFrame(audit["real_strategy_records"])
    .filter(pl.col("framework").is_in(FRAMEWORKS))
    .with_columns(
        pl.col("case_study").replace_strict(CASE_NAMES).alias("strategy"),
        pl.col("framework").replace_strict(FRAMEWORK_NAMES).alias("engine"),
    )
    .select(
        "strategy",
        "engine",
        "status",
        "fills",
        "valuations",
        "valuation_timestamps_match",
        "equity_gap",
        "equity_raw_gap",
        "terminal_gap",
        "terminal_raw_gap",
    )
    .sort("strategy", "engine")
)

assert results.height == 7
assert results.filter(pl.col("status") == "pass").height == 7
assert results["valuation_timestamps_match"].all()

display(results)

strategy,engine,status,fills,valuations,valuation_timestamps_match,equity_gap,equity_raw_gap,terminal_gap,terminal_raw_gap
str,str,str,i64,i64,bool,str,str,str,str
"""CME futures""","""VectorBT Pro 2026.6.27""","""pass""",3545,1595,true,"""0.00""","""0.00000010""","""0.00""","""0.00000007"""
"""ETF allocation""","""VectorBT OSS 1.1.0""","""pass""",2466,1995,true,"""0.00""","""0.00000000""","""0.00""","""0.00000000"""
"""ETF allocation""","""VectorBT Pro 2026.6.27""","""pass""",2466,1995,true,"""0.00""","""0.00000000""","""0.00""","""0.00000000"""
"""FX allocation (USD-quoted pair…","""VectorBT OSS 1.1.0""","""pass""",279,2108,true,"""0.00""","""0.00000000""","""0.00""","""0.00000000"""
"""FX allocation (USD-quoted pair…","""VectorBT Pro 2026.6.27""","""pass""",279,2108,true,"""0.00""","""0.00000000""","""0.00""","""0.00000000"""
"""US equity panel""","""VectorBT OSS 1.1.0""","""pass""",49120,4146,true,"""0.00""","""0.00001910""","""0.00""","""0.00001870"""
"""US equity panel""","""VectorBT Pro 2026.6.27""","""pass""",54295,4146,true,"""0.00""","""0.00001950""","""0.00""","""0.00001880"""


Both VectorBT editions participate in the ETF, USD-quoted foreign-exchange, and US equity-panel
comparisons. VectorBT Pro also supplies contract multipliers and futures-style leverage for the
CME comparison. Fill prices retain eight-decimal precision, quantities retain five-decimal
precision, and account monetary values must round to the same cent.

<!-- ML4T_ZH_BEGIN c0007-p000 -->
兩個 VectorBT 版本都參與 ETF、以美元報價的外匯，以及美國股票資料面板的比較。VectorBT Pro 另外提供 CME 比較所需的契約乘數與期貨式槓桿。成交價格保留小數點後八位精度，數量保留小數點後五位精度，帳戶貨幣價值則須四捨五入至相同的分。
<!-- ML4T_ZH_END c0007-p000 -->


## 2. Unsupported asset models

<!-- ML4T_ZH_BEGIN c0008-p000 -->
## 2. 不支援的資產模型
<!-- ML4T_ZH_END c0008-p000 -->


In [5]:
unsupported = (
    pl.DataFrame(audit["unsupported_records"])
    .filter(pl.col("framework").is_in(FRAMEWORKS))
    .with_columns(
        pl.col("case_study").replace_strict(CASE_NAMES).alias("strategy"),
        pl.col("framework").replace_strict(FRAMEWORK_NAMES).alias("engine"),
    )
    .select("strategy", "engine", "reason")
    .sort("strategy", "engine")
)
display(unsupported)

strategy,engine,reason
str,str,str
"""CME futures""","""VectorBT OSS 1.1.0""","""no native futures contract mul…"
"""Crypto perpetual funding""","""VectorBT OSS 1.1.0""","""no native perpetual-futures fu…"
"""Crypto perpetual funding""","""VectorBT Pro 2026.6.27""","""no native perpetual-futures fu…"


VectorBT OSS does not provide the native multiplier and margin-account model required by the CME
bundle. Neither edition is used to emulate crypto-perpetual funding and margin accounting.

<!-- ML4T_ZH_BEGIN c0010-p000 -->
VectorBT 開源版未提供 CME 資料組合所需的原生乘數與保證金帳戶模型。兩個版本都不會用來模擬加密貨幣永續契約的資金費率與保證金會計。
<!-- ML4T_ZH_END c0010-p000 -->


## 3. Engine-only timing

Each correctness-passing VectorBT row has an engine-only timing record.

<!-- ML4T_ZH_BEGIN c0011-p000 -->
## 3. 僅計引擎的執行時間

每列通過正確性檢驗的 VectorBT 結果，都有一筆僅計引擎的計時紀錄。
<!-- ML4T_ZH_END c0011-p000 -->


In [6]:
timing = (
    pl.DataFrame(audit["performance_records"])
    .filter(pl.col("framework").is_in(FRAMEWORKS))
    .with_columns(
        pl.col("case_study").replace_strict(CASE_NAMES).alias("strategy"),
        pl.col("framework").replace_strict(FRAMEWORK_NAMES).alias("engine"),
        pl.col("framework_median_seconds").round(ROUND_SECONDS).alias("vectorbt_seconds"),
        pl.col("ml4t_median_seconds").round(ROUND_SECONDS).alias("ml4t_seconds"),
        pl.col("framework_to_ml4t_ratio").round(2).alias("vectorbt_div_ml4t"),
    )
    .select("strategy", "engine", "vectorbt_seconds", "ml4t_seconds", "vectorbt_div_ml4t")
)

assert timing.height == 7
display(timing)

strategy,engine,vectorbt_seconds,ml4t_seconds,vectorbt_div_ml4t
str,str,f64,f64,f64
"""ETF allocation""","""VectorBT Pro 2026.6.27""",0.287,0.415,0.69
"""ETF allocation""","""VectorBT OSS 1.1.0""",0.171,0.415,0.41
"""CME futures""","""VectorBT Pro 2026.6.27""",0.286,0.432,0.66
"""FX allocation (USD-quoted pair…","""VectorBT Pro 2026.6.27""",0.283,0.146,1.94
"""FX allocation (USD-quoted pair…","""VectorBT OSS 1.1.0""",0.139,0.146,0.95
"""US equity panel""","""VectorBT Pro 2026.6.27""",0.795,20.555,0.04
"""US equity panel""","""VectorBT OSS 1.1.0""",16.995,20.495,0.83


The measured region excludes data loading, target construction, adapter preparation, and output
extraction. The observed ratios do not establish the same relationship for other datasets,
strategy mechanics, or machines.

<!-- ML4T_ZH_BEGIN c0013-p000 -->
測量區間不包含資料載入、目標建構、轉接器準備及輸出擷取。觀測到的比值無法證明其他資料集、策略機制或機器具有相同的速度關係。
<!-- ML4T_ZH_END c0013-p000 -->


## 4. Synthetic stress evidence

<!-- ML4T_ZH_BEGIN c0014-p000 -->
## 4. 合成壓力測試證據
<!-- ML4T_ZH_END c0014-p000 -->


In [7]:
stress = (
    pl.DataFrame(audit["synthetic_stress"]["records"])
    .filter(pl.col("framework").is_in(FRAMEWORKS))
    .with_columns(pl.col("framework").replace_strict(FRAMEWORK_NAMES).alias("engine"))
    .select("engine", "intents", "fills", "trades", "terminal_value", "status")
)
display(stress)

engine,intents,fills,trades,terminal_value,status
str,i64,i64,i64,f64,str
"""VectorBT Pro 2026.6.27""",427790,423313,222751,1.2859e6,"""pass"""
"""VectorBT OSS 1.1.0""",427790,417941,211322,1.3453e6,"""pass"""


VectorBT Pro and OSS both pass the generated stress comparison against their matching ML4T
profiles. This establishes scale conformance for the fixed target-order recipe. It does not create
support for the excluded asset models.

<!-- ML4T_ZH_BEGIN c0016-p000 -->
VectorBT Pro 與開源版均通過與各自對應 ML4T 設定組合的生成資料壓力測試比較。這證明固定目標委託規則在規模上的一致性，但不代表因此支援原本排除的資產模型。
<!-- ML4T_ZH_END c0016-p000 -->
